Load Libraries

In [7]:
import duckdb
import pandas as pd

from pathlib import Path

from detector.eval.metrics import summarise, threshold_at_fpr
from detector.eval.stress import leave_one_topic_out, truncate_words
from detector.features.augment import add_truncated_copies, truncate_random
from detector.models.tfidf import fit, score


In [9]:
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
df = duckdb.connect().sql(f"select * from '{ROOT / 'data/processed/essays.parquet'}'").df()
train, val = df[df.split == "train"], df[df.split == "val"]

In [10]:
aug = add_truncated_copies(train, n_copies=2, seed=0, min_words=20)
print(len(train), len(aug))
models = {"baseline": fit(train.text, train.label), "augmented": fit(aug.text, aug.label)}

14171 42513


In [11]:
mixed = truncate_random(val, seed=123, min_words=20)
thresholds = {}
for name, (v, c) in models.items():
    s = score(v, c, mixed.text)
    thresholds[name] = threshold_at_fpr(mixed.label, s, 0.01)
    print(name, summarise(mixed.label, s, thresholds[name]))

baseline {'roc_auc': 0.9969592089788822, 'pr_auc': 0.9881246903641182, 'tpr': 0.9552845528455285, 'fpr': 0.009827044025157232}
augmented {'roc_auc': 0.9994279541852022, 'pr_auc': 0.9975531459160125, 'tpr': 0.9878048780487805, 'fpr': 0.009827044025157232}


In [13]:
rows = []
for name, (v, c) in models.items():
    for n in (20, 30, 45, 60, 100, 200, None):
        texts = val.text if n is None else val.text.map(lambda x: truncate_words(x, n))
        rows.append({"model": name, "words": n or "full", **summarise(val.label, score(v, c, texts), thresholds[name])})
pd.DataFrame(rows).round(3)

,model,words,roc_auc,pr_auc,tpr,fpr
0,baseline,20,0.961,0.895,0.846,0.040
1,baseline,30,0.976,0.933,0.878,0.027
2,baseline,45,0.991,0.975,0.917,0.015
3,baseline,60,0.998,0.991,0.955,0.007
4,baseline,100,1.000,0.999,0.980,0.001
5,baseline,200,1.000,1.000,0.992,0.000
6,baseline,full,1.000,1.000,0.996,0.000
7,augmented,20,0.994,0.980,0.965,0.038
8,augmented,30,0.998,0.992,0.976,0.024
9,augmented,45,0.999,0.997,0.990,0.015


In [14]:
plain = leave_one_topic_out(df)
robust = leave_one_topic_out(df, augment=lambda d: add_truncated_copies(d, 2, 0, 20))
cols = ["held_out_topic", "roc_auc", "tpr", "fpr"]
plain[cols].merge(robust[cols], on="held_out_topic", suffixes=("_base", "_aug")).round(3)

,held_out_topic,roc_auc_base,tpr_base,fpr_base,roc_auc_aug,tpr_aug,fpr_aug
0,0,0.999,1.000,0.113,0.997,0.964,0.020
1,1,0.999,0.953,0.004,0.997,0.844,0.003
2,2,0.985,0.913,0.033,0.982,0.867,0.019
3,3,0.993,0.308,0.000,0.996,0.234,0.000
4,4,0.977,0.407,0.000,0.989,0.499,0.000
5,5,0.999,0.991,0.000,1.000,0.966,0.000
6,6,0.996,0.950,0.014,0.993,0.936,0.016
